In [1]:
# config.py
UC_HOST = "http://localhost:8080/api/2.1/unity-catalog"

# ── Volume storage: plain directories for unstructured files ───────────────
# UC Volumes in OSS are metadata registrations pointing to a path.
# In production these would be s3://, abfss://, or gs:// URIs.
# We keep a local filesystem path (for os/shutil) and a file:// URI (for UC API)
# as separate variables — they represent the same location in two different contexts.
VOLUMES_ROOT     = "/opt/unity/data/uc_volumes/docs_catalog/ingestion"
VOLUMES_ROOT_URI = "file:///opt/unity/data/uc_volumes/docs_catalog/ingestion"
RAW_DOCS_DIR     = f"{VOLUMES_ROOT}/raw_docs"
RAW_DOCS_URI     = f"{VOLUMES_ROOT_URI}/raw_docs"

# ── Table storage: Delta tables, each with its own _delta_log ─────────────
# Completely separate root from volumes.
TABLES_ROOT       = "/opt/unity/data/uc_tables/docs_catalog"
CHUNKS_PATH       = f"file://{TABLES_ROOT}/ingestion/parsed_chunks"
EMBEDDINGS_PATH   = f"file://{TABLES_ROOT}/ingestion/embedded_chunks"
WATERMARK_PATH    = f"file://{TABLES_ROOT}/retrieval/chunk_index_watermark"
# Note: DeltaTable and write_deltalake accept file:// URIs directly.
# For local filesystem reads (e.g. DuckDB), strip the file:// prefix.

MLFLOW_URI    = "http://localhost:5000"
MLFLOW_UC_URI = "uc:http://localhost:8080"   # UC as MLflow model registry

OS_HOST  = "localhost"
OS_PORT  = 9200
OS_INDEX = "doc_chunks"

In [2]:
# 00_setup_uc.py
import asyncio, os
from unitycatalog.client import ApiClient, Configuration
from unitycatalog.client.api import VolumesApi, TablesApi
from unitycatalog.client.models import (
    CreateVolumeRequestContent, VolumeType,
    CreateTable, TableType, DataSourceFormat, ColumnInfo, ColumnTypeName,
)
from unitycatalog.ai.core.client import UnitycatalogClient

config     = Configuration(host=UC_HOST)
api_client = ApiClient(configuration=config)
uc         = UnitycatalogClient(api_client=api_client)

# create_catalog / create_schema are idempotent: they warn and return
# the existing object if the name already exists.
uc.create_catalog(name="docs_catalog", comment="Document intelligence pipeline")
uc.create_schema(name="ingestion", catalog_name="docs_catalog")
uc.create_schema(name="retrieval", catalog_name="docs_catalog")

# Create the local directory before registering the Volume.
# UC does not create the directory — it only stores the metadata pointer.
#os.makedirs(RAW_DOCS_DIR, exist_ok=True)

async def register_volume():
    volumes_api = VolumesApi(api_client=api_client)
    try:
        await volumes_api.create_volume(CreateVolumeRequestContent(
            catalog_name="docs_catalog",
            schema_name="ingestion",
            name="raw_docs",
            volume_type=VolumeType.EXTERNAL,
            storage_location=RAW_DOCS_URI,   # file:// URI, not a filesystem path
            comment="Raw PDF files. Write-once source of truth.",
        ))
        print("Volume registered.")
    except Exception as e:
        print(f"Volume already registered or error: {e}")

asyncio.run(register_volume())

Volume registered.
